# DataCleanEnv — RL Training with GRPO

This notebook trains an LLM to clean messy data using GRPO (Group Relative Policy Optimization) from the `trl` library.

## Setup
1. Go to Runtime → Change runtime type → GPU (T4 or better)
2. Run all cells in order
3. Monitor training progress

## Hardware Requirements
- **T4 (16GB)**: Can train Qwen2.5-3B with 4-bit quantization
- **A100 (40GB)**: Can train Qwen2.5-7B or larger
- **L4 (24GB)**: Good balance of speed and memory

In [1]:
!nvidia-smi

Sun Apr  5 05:34:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Clone Repository & Install Dependencies

In [2]:
# Clone the repository
!git clone https://github.com/tanmaydesai07/openenv-datacleaning-agent.git
%cd openenv-datacleaning-agent

# Install dependencies
!pip install -q nest_asyncio trl transformers accelerate peft bitsandbytes
!pip install -q pandas numpy fastapi fastmcp uvicorn websockets httpx
!pip install -q openenv-core

# Install the data_clean_env package
!pip install -e data_clean_env/

Cloning into 'openenv-datacleaning-agent'...
remote: Enumerating objects: 1218, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 1218 (delta 11), reused 45 (delta 9), pack-reused 1166 (from 1)
Receiving objects: 100% (1218/1218), 60.29 MiB | 18.27 MiB/s, done.
Resolving deltas: 100% (191/191), done.
/content/openenv-datacleaning-agent
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 2.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 705.5/705.5 kB 19.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.0 MB/s eta 0:00:00
   ━

## 2. Check GPU & Configuration

In [ ]:
import torch
import os

# Check GPU
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Auto-select model based on GPU memory
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
if gpu_memory >= 40:  # A100
    MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
elif gpu_memory >= 20:  # L4
    MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
else:  # T4 (16GB)
    MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"  # Still works with 4-bit!

print(f"Selected model: {MODEL_ID}")

# ============================================
# 🔑 HUGGINGFACE TOKEN - Set your token here!
# ============================================
# Get your token from: https://huggingface.co/settings/tokens
HF_TOKEN = ""  # <-- PASTE YOUR TOKEN HERE

# Set as environment variable FIRST (prevents Colab secret fetch warning)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# Login to HuggingFace
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f"✅ Logged in to HuggingFace Hub")
else:
    print("⚠️ No HF_TOKEN set - set it above for faster downloads")

# Training configuration
CONFIG = {
    "learning_rate": 2e-5,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "max_prompt_length": 1024,
    "max_completion_length": 512,
    "num_generations": 2,
    "beta": 0.1,
    "temperature": 0.7,
    "output_dir": "./dataclean-grpo-checkpoint",
    "logging_steps": 1,
    "save_steps": 25,
}

GPU: Tesla T4
VRAM: 15.6 GB
Selected model: Qwen/Qwen2.5-3B-Instruct


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Logged in to HuggingFace Hub


## 3. Generate Diverse Training Data

In [4]:
# Generate diverse domain-specific datasets for better generalization
!python generate_training_data.py --output-dir data_clean_env/tasks --num-samples 50 --num-datasets 5

# List generated files
import os
tasks_dir = "data_clean_env/tasks"
files = sorted(os.listdir(tasks_dir))
print(f"\n✅ Available task files ({len(files)} total):")
for f in files:
    size = os.path.getsize(os.path.join(tasks_dir, f))
    print(f"   {f} ({size} bytes)")

Generating 5 dataset pairs with 50 rows each...
  ✅ hr: 54 messy rows → 50 clean rows
     Messy: data_clean_env/tasks/hr_messy.csv
     Clean: data_clean_env/tasks/hr_clean.csv
  ✅ healthcare: 54 messy rows → 50 clean rows
     Messy: data_clean_env/tasks/healthcare_messy.csv
     Clean: data_clean_env/tasks/healthcare_clean.csv
  ✅ finance: 51 messy rows → 50 clean rows
     Messy: data_clean_env/tasks/finance_messy.csv
     Clean: data_clean_env/tasks/finance_clean.csv
  ✅ logistics: 51 messy rows → 50 clean rows
     Messy: data_clean_env/tasks/logistics_messy.csv
     Clean: data_clean_env/tasks/logistics_clean.csv
  ✅ education: 52 messy rows → 50 clean rows
     Messy: data_clean_env/tasks/education_messy.csv
     Clean: data_clean_env/tasks/education_clean.csv

Done! Generated 5 dataset pairs in data_clean_env/tasks

✅ Available task files (16 total):
   easy_clean.csv (161 bytes)
   easy_messy.csv (198 bytes)
   education_clean.csv (2736 bytes)
   education_messy.csv (2881 byt

## 4. Start Environment Server

In [5]:
import subprocess
import time

# Start the FastAPI server in background
server_process = subprocess.Popen(
    ["uvicorn", "data_clean_env.server.app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for server to start
time.sleep(8)
print("✅ Environment server started on http://localhost:8000")

# Test health endpoint
import urllib.request
try:
    response = urllib.request.urlopen("http://localhost:8000/health", timeout=5)
    print(f"✅ Health check passed: {response.status}")
except Exception as e:
    print(f"⚠️ Server starting... will retry: {e}")
    time.sleep(5)
    response = urllib.request.urlopen("http://localhost:8000/health", timeout=5)
    print(f"✅ Health check passed: {response.status}")

✅ Environment server started on http://localhost:8000
✅ Health check passed: 200


## 5. Create Training Dataset

In [6]:
from datasets import Dataset
import os

# System prompt for data cleaning
SYSTEM_PROMPT = """You are an expert data cleaning agent. Clean messy CSV files by:
1. Reading the file with read_file(path)
2. Using run_python(code) to clean with pandas (remove duplicates, fix missing values, normalize dates, trim whitespace, etc.)
3. Submitting with submit_cleaned_file(path='cleaned_output.csv')

Always save cleaned data with: df.to_csv('cleaned_output.csv', index=False)"""

# Training prompts covering different domains and difficulty levels
prompts = [
    # Easy - customer data
    f"{SYSTEM_PROMPT}\n\nTask: Clean customer dataset. Remove duplicates and handle missing values. File: easy_messy.csv",
    f"{SYSTEM_PROMPT}\n\nTask: Clean this dataset with duplicate rows and missing emails. File: easy_messy.csv",
    
    # Medium - sales records  
    f"{SYSTEM_PROMPT}\n\nTask: Clean sales records. Fix: inconsistent dates, mixed casing, invalid emails, extra whitespace. File: medium_messy.csv",
    f"{SYSTEM_PROMPT}\n\nTask: Normalize dates to YYYY-MM-DD, title case names, validate emails, convert amounts to float. File: medium_messy.csv",
    
    # Hard - product inventory
    f"{SYSTEM_PROMPT}\n\nTask: Clean product inventory. Fix: missing IDs, negative quantities, invalid prices, inconsistent formats. File: hard_messy.csv",
    f"{SYSTEM_PROMPT}\n\nTask: Generate missing IDs, clamp negatives to 0, normalize all data. File: hard_messy.csv",
]

# Add domain-specific prompts if files exist
domain_prompts = [
    # HR domain
    ("hr_messy.csv", "Clean HR employee records. Fix: name casing, salary formats ($, USD), date formats, email issues, missing values."),
    ("hr_messy.csv", "Standardize employee data: title case names, numeric salaries, YYYY-MM-DD dates, valid lowercase emails."),
    
    # Healthcare domain
    ("healthcare_messy.csv", "Clean patient records. Fix: name casing, condition casing, date formats, doctor name formats, status casing."),
    ("healthcare_messy.csv", "Normalize healthcare data: title case all names, standardize dates, fill missing insurance IDs."),
    
    # Finance domain
    ("finance_messy.csv", "Clean financial transactions. Fix: merchant casing, amount formats ($, USD), date formats, category casing."),
    ("finance_messy.csv", "Standardize transaction data: title case merchants, numeric amounts, YYYY-MM-DD dates."),
    
    # Logistics domain
    ("logistics_messy.csv", "Clean shipping records. Fix: origin/destination casing, carrier names, date formats, status casing."),
    ("logistics_messy.csv", "Normalize logistics data: title case locations, standardize dates, fix weight formats."),
    
    # Education domain
    ("education_messy.csv", "Clean student records. Fix: name casing, subject casing, date formats, status casing, missing credits."),
    ("education_messy.csv", "Standardize grade records: title case names/subjects, YYYY-MM-DD dates, fill missing credits."),
]

tasks_dir = "data_clean_env/tasks"
for filename, task_desc in domain_prompts:
    if os.path.exists(os.path.join(tasks_dir, filename)):
        prompts.append(f"{SYSTEM_PROMPT}\n\nTask: {task_desc} File: {filename}")

# Repeat prompts for more training samples
train_dataset = Dataset.from_dict({"prompt": prompts * 2})  # ~32 samples for fast training
print(f"✅ Created training dataset with {len(train_dataset)} samples")
print(f"   Unique prompts: {len(prompts)}")
print(f"   Domains covered: easy, medium, hard + HR, healthcare, finance, logistics, education")

✅ Created training dataset with 32 samples
   Unique prompts: 16
   Domains covered: easy, medium, hard + HR, healthcare, finance, logistics, education


## 6. Define Reward Function (Using Real Environment)

In [7]:
import re

def reward_fn(completions, **kwargs):
    """Heuristic reward function for GRPO training.
    
    Rewards the model for:
    1. Using the correct tool sequence (read → run_python → submit)
    2. Saving cleaned data with df.to_csv()
    3. Submitting the correct file path
    
    This is the standard approach for tool-calling agent training.
    """
    rewards = []
    
    for completion in completions:
        if isinstance(completion, list):
            text = " ".join([t.get("text", str(t)) for t in completion])
        else:
            text = str(completion)
        
        reward = 0.0
        
        # Reward for reading the file (first step)
        if "read_file" in text:
            reward += 0.15
        
        # Reward for using pandas to clean data
        if "run_python" in text:
            reward += 0.25
            if "pandas" in text or "import pd" in text:
                reward += 0.1
        
        # Reward for saving cleaned data
        if "to_csv" in text and "cleaned_output.csv" in text:
            reward += 0.2
        
        # Reward for submitting the correct file
        if "submit_cleaned_file" in text:
            reward += 0.2
            if "cleaned_output.csv" in text:
                reward += 0.1
        
        # Bonus for proper cleaning operations
        cleaning_ops = ["drop_duplicates", "fillna", "strip", "to_datetime", "str.lower", "str.upper", "str.title"]
        ops_found = sum(1 for op in cleaning_ops if op in text)
        reward += min(0.1, ops_found * 0.02)
        
        # Cap at 1.0
        reward = min(1.0, reward)
        
        rewards.append(reward)
    
    return rewards

print("✅ Reward function defined (heuristic-based for fast training)")

✅ Reward function defined (heuristic-based for fast training)


## 7. Load Model and Tokenizer

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="flash_attention_2" if torch.cuda.get_device_capability()[0] >= 8 else "eager",
)

print(f"✅ Model loaded: {MODEL_ID}")
print(f"✅ Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"✅ Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model loaded: Qwen/Qwen2.5-3B-Instruct
✅ Parameters: 3.09B
✅ Memory used: 2.07 GB


## 8. Setup GRPO Trainer

In [9]:
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

# LoRA for memory-efficient training
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

# GRPO training config (TRL 1.0+)
# See: https://huggingface.co/docs/trl/grpo_trainer
training_args = GRPOConfig(
    output_dir=CONFIG["output_dir"],
    learning_rate=CONFIG["learning_rate"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    num_generations=CONFIG["num_generations"],
    max_completion_length=CONFIG["max_completion_length"],
    temperature=CONFIG["temperature"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    bf16=True,
    optim="adamw_torch_fused",
    gradient_checkpointing=True,
    warmup_steps=10,
)

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=lora_config,
)

print("✅ GRPO Trainer initialized")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")

✅ GRPO Trainer initialized
   Trainable params: 29.9M


## 9. Train the Model

In [ ]:
print("🚀 Starting GRPO training...")
print(f"   Model: {MODEL_ID}")
print(f"   Training samples: {len(train_dataset)}")
print(f"   Epochs: {CONFIG['num_train_epochs']}")
print(f"   Effective batch size: {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"   Generations per prompt: {CONFIG['num_generations']}")
print()

# Train!
train_result = trainer.train()

# Save the final model
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"\n✅ Training complete!")
print(f"✅ Model saved to {CONFIG['output_dir']}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Starting GRPO training...
   Model: Qwen/Qwen2.5-3B-Instruct
   Training samples: 32
   Epochs: 1
   Effective batch size: 4
   Generations per prompt: 2



Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000
6,0.000000
7,-0.000000
8,0.000001
9,0.000000
10,-0.000000


## 10. Evaluate the Trained Model

In [ ]:
from transformers import pipeline
import torch

# Merge LoRA weights and create inference pipeline
merged_model = trainer.model.merge_and_unload()

pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Test on sample
test_prompt = """You are an expert data cleaning agent. Clean messy CSV files by:
1. Reading the file with read_file(path)
2. Using run_python(code) to clean with pandas
3. Submitting with submit_cleaned_file(path='cleaned_output.csv')

Task: Clean customer dataset with duplicates and missing values. File: easy_messy.csv"""

output = pipe(
    test_prompt,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
    return_full_text=False,
)

print("📝 Generated response:")
print(output[0]["generated_text"])

## 11. Push to HuggingFace Hub (Optional)

In [ ]:
# # Push trained model to HuggingFace Hub
# # Make sure HF_TOKEN is set in Cell 2!

# HF_USERNAME = "your-username"  # <-- Change to your HuggingFace username!
# MODEL_NAME = "dataclean-agent-grpo"

# if HF_TOKEN:
#     merged_model.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}", token=HF_TOKEN)
#     tokenizer.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}", token=HF_TOKEN)
#     print(f"✅ Model pushed to https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")
# else:
#     print("❌ HF_TOKEN not set - go back to Cell 2 and set your token")

## 12. Cleanup

In [ ]:
# Stop the environment server
server_process.terminate()
server_process.wait()
print("✅ Environment server stopped")